# Create boundary from a Steve mosaic

Derives tissue boundaries (`boundary_positions*.txt`) and holes (`hole*.txt`)
from a low-mag Steve mosaic (e.g. `data/mosaic10x/`), instead of drawing them
by hand. Writes them to `positions/boundaries/from_mosaic/`, which
`03_create_positions_from_boundaries.ipynb` reads (its `BOUNDARY_SOURCE`).

**Method:** the tiles are pasted into one image at their stage positions,
smoothed, thresholded and cleaned up (closing, opening, outward margin).
Background enclosed by tissue becomes a hole; tissue islands inside a hole
stay tissue. Both are traced into polygons in stage microns. Details in
`MERci.acquisition.mosaic`.

**Iterative:** re-run the segmentation and plot cells with different
parameters until the green (tissue) / red (hole) overlay looks right, then run
the write cell.

In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/before_imaging/multi_z/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity
from MERci.acquisition.mosaic import (
    load_steve_mosaic, assemble_mosaic_canvas,
    segment_mosaic_tissue, save_boundary_from_mosaic,
)
from MERci.plots.mosaic_plots import (
    plot_tile_intensity_histograms, plot_mosaic_segmentation,
    plot_objective_alignment_check,
)

POSITIONS_DIR = SAMPLE_DIR / "positions"
OUTPUT_DIR = POSITIONS_DIR / "boundaries" / "from_mosaic"   # hand-drawn ones: boundaries/manual/
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Printed only to confirm the expected experiment was found.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)

print(f"SAMPLE_DIR : {SAMPLE_DIR}")
print(f"SAMPLE_NAME: {SAMPLE_NAME}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

## Load the mosaic

Steve's "Save Mosaic" writes an `.msc` manifest plus `.stv` tiles. Set
`MOSAIC_DIR`/`MOSAIC_NAME` if yours is elsewhere or named differently.

In [ ]:
MOSAIC_DIR  = SAMPLE_DIR / "data" / "mosaic10x"
MOSAIC_NAME = None   # e.g. "mosaic10x" -- None = auto-detect the only *.msc file present

msc_candidates = sorted(MOSAIC_DIR.glob(f"{MOSAIC_NAME or '*'}.msc"))

# No mosaic yet: use a bundled example canvas from a past experiment, so this
# notebook and 03 can be tested before a real scan exists.
tiles_all, tiles = [], []
MAJORITY_OBJECTIVE   = None
USING_EXAMPLE_MOSAIC = not msc_candidates

if USING_EXAMPLE_MOSAIC:
    from MERci.acquisition.mosaic import load_mosaic_canvas
    EXAMPLE_CANVAS_PATH = MERCI_DIR / "data" / "mosaic_canvas_examples" / "lineage_tracing" / "canvas.npz"
    if not EXAMPLE_CANVAS_PATH.exists():
        raise FileNotFoundError(
            f"No .msc mosaic manifest found in {MOSAIC_DIR}, and no bundled "
            f"example canvas at {EXAMPLE_CANVAS_PATH} either -- copy Steve's "
            f"saved mosaic to {MOSAIC_DIR} (or set MOSAIC_DIR/MOSAIC_NAME above)."
        )
    canvas = load_mosaic_canvas(EXAMPLE_CANVAS_PATH)
    print(f"WARNING: no .msc mosaic manifest in {MOSAIC_DIR}; using the bundled "
          f"example canvas instead ({EXAMPLE_CANVAS_PATH}), from a past real experiment.")
    print(f"Canvas: {canvas.image.shape[1]}x{canvas.image.shape[0]} px "
          f"at {canvas.pixel_size_um:.2f} um/px, origin {canvas.origin_um}")
else:
    if len(msc_candidates) > 1:
        print(f"WARNING: {len(msc_candidates)} .msc files found, using the first: "
              f"{msc_candidates[0].name}. Set MOSAIC_NAME to pick a different one.")
    MSC_PATH = msc_candidates[0]

    tiles_all = load_steve_mosaic(MSC_PATH)
    print(f"Loaded {len(tiles_all)} tile(s) from {MSC_PATH.name}")

    obj_counts = Counter(t.objective_name for t in tiles_all)
    print(f"Objective breakdown: {dict(obj_counts)}")

    # Objectives to composite. Higher-mag reference tiles (e.g. "60x") have a
    # different exposure and confuse thresholding. None = keep all.
    MOSAIC_KEEP_OBJECTIVES = ["10x"]

    tiles = tiles_all if MOSAIC_KEEP_OBJECTIVES is None else [
        t for t in tiles_all if t.objective_name in MOSAIC_KEEP_OBJECTIVES
    ]
    n_excluded = len(tiles_all) - len(tiles)
    print(f"Using {len(tiles)} tile(s)"
          + (f" (excluded {n_excluded} not in {MOSAIC_KEEP_OBJECTIVES})" if n_excluded else ""))

    # The threshold histogram below uses only this objective's tiles.
    kept_obj_counts = Counter(t.objective_name for t in tiles)
    MAJORITY_OBJECTIVE = max(kept_obj_counts, key=kept_obj_counts.get)

    WORKING_PIXEL_UM = 5.0   # canvas resolution -- smaller = sharper but slower/more memory
    canvas = assemble_mosaic_canvas(tiles, working_pixel_um=WORKING_PIXEL_UM)
    print(f"Canvas: {canvas.image.shape[1]}x{canvas.image.shape[0]} px "
          f"at {canvas.pixel_size_um:.2f} um/px, origin {canvas.origin_um}")

## Verify objective alignment (optional)

For a mosaic with several objectives: overlays them, each normalized
separately, to check that they line up. Diagnostic only, does not affect the
segmentation. Needs real tiles (not the example canvas).

In [ ]:
VERIFY_OBJECTIVE_ALIGNMENT = False   # True to render the overlay

if VERIFY_OBJECTIVE_ALIGNMENT:
    if USING_EXAMPLE_MOSAIC:
        print("Using the bundled example canvas (no raw tiles available) -- "
              "objective-alignment verification needs a real multi-objective mosaic. Skipping.")
    else:
        objectives_present = sorted(obj_counts)
        if len(objectives_present) < 2:
            print(f"Only one objective present ({objectives_present}) -- nothing to corroborate. Skipping.")
        else:
            ax, _ = plot_objective_alignment_check(tiles_all, working_pixel_um=WORKING_PIXEL_UM)
            ax.figure.set_size_inches(10, 10)
            ax.figure.tight_layout()
            plt.show()

## Inspect intensity to pick a threshold

Log-intensity histograms of each `MAJORITY_OBJECTIVE` tile (gray) and of all
of them pooled (black). If the pooled histogram is bimodal, the valley between
the background and tissue peaks (red dashed line) is the starting
`THRESHOLD`. Other objectives are left out, since they add an extra peak.

In [ ]:
if USING_EXAMPLE_MOSAIC:
    # The example canvas has no raw tiles; 400 is its tuned threshold.
    ESTIMATED_THRESHOLD = 400
    print(f"Using the bundled example canvas -- skipping the per-tile histogram "
          f"estimate (no raw tiles available); ESTIMATED_THRESHOLD="
          f"{ESTIMATED_THRESHOLD} (this canvas's own real tuned value).")
else:
    tiles_for_threshold = [t for t in tiles if t.objective_name == MAJORITY_OBJECTIVE]

    ax_hist, ESTIMATED_THRESHOLD = plot_tile_intensity_histograms(tiles_for_threshold)
    ax_hist.figure.set_size_inches(8, 5)
    ax_hist.figure.tight_layout()

    print(f"Estimated threshold from {len(tiles_for_threshold)} '{MAJORITY_OBJECTIVE}' tile(s) "
          f"(valley between background/tissue peaks): {ESTIMATED_THRESHOLD}")

## Segment tissue + holes -- tune and re-run

Re-run this cell and the plot below until the overlay looks right.
`THRESHOLD` starts at `ESTIMATED_THRESHOLD`. Set a number to override it, or
`None` for Otsu. Distances are in microns, independent of `WORKING_PIXEL_UM`.

- `SMOOTH_SIGMA_UM` -- blur before thresholding. Too small: fragmented mask.
  Too large: loses fine edges.
- `CLOSE_RADIUS_UM` -- bridges small gaps, so nearby bits merge into one piece.
- `OPEN_RADIUS_UM` -- removes small specks. Too large erodes thin tissue.
- `MARGIN_UM` -- outward buffer added to the boundary.
- `MIN_TISSUE_AREA_UM2`/`MIN_HOLE_AREA_UM2` -- drop smaller pieces/holes.
- `MIN_ISLAND_AREA_UM2` -- tissue islands inside a hole at least this big stay
  tissue (dashed in the plot).
- `BOUNDARY_MAX_DEVIATION_UM` -- how far the simplified boundary may stray
  from the traced contour. Larger = fewer vertices.

In [ ]:
if USING_EXAMPLE_MOSAIC:
    # Tuned for the example canvas (the defaults below fragment it).
    SMOOTH_SIGMA_UM     = 10.0
    CLOSE_RADIUS_UM     = 50.0
    OPEN_RADIUS_UM      = 15.0
    MARGIN_UM           = 25.0
    MIN_TISSUE_AREA_UM2 = 4_000_000.0
    MIN_HOLE_AREA_UM2   = 12_000.0
    MIN_ISLAND_AREA_UM2 = 50_000.0
    BOUNDARY_MAX_DEVIATION_UM = 15.0
else:
    SMOOTH_SIGMA_UM     = 10.0
    CLOSE_RADIUS_UM     = 50.0
    OPEN_RADIUS_UM      = 15.0
    MARGIN_UM           = 25.0
    MIN_TISSUE_AREA_UM2 = 1000.0
    MIN_HOLE_AREA_UM2   = 500.0
    MIN_ISLAND_AREA_UM2 = 1000.0
    BOUNDARY_MAX_DEVIATION_UM = 15.0

# > 0: keep pieces below MIN_TISSUE_AREA_UM2 that lie within this distance of
# a kept piece. 0 = drop them all.
NEAR_FRAGMENT_MAX_DISTANCE_UM = 0.0
MIN_NEAR_FRAGMENT_AREA_UM2    = 0.0   # size floor for those pieces

THRESHOLD = ESTIMATED_THRESHOLD   # None = Otsu

segmentation = segment_mosaic_tissue(
    canvas,
    threshold           = THRESHOLD,
    smooth_sigma_um     = SMOOTH_SIGMA_UM,
    close_radius_um     = CLOSE_RADIUS_UM,
    open_radius_um      = OPEN_RADIUS_UM,
    margin_um           = MARGIN_UM,
    min_tissue_area_um2 = MIN_TISSUE_AREA_UM2,
    min_hole_area_um2   = MIN_HOLE_AREA_UM2,
    min_island_area_um2 = MIN_ISLAND_AREA_UM2,
    boundary_max_deviation_um = BOUNDARY_MAX_DEVIATION_UM,
    near_fragment_max_distance_um = NEAR_FRAGMENT_MAX_DISTANCE_UM,
    min_near_fragment_area_um2    = MIN_NEAR_FRAGMENT_AREA_UM2,
)
print(f"Threshold used : {segmentation.threshold:.1f}")
print(f"Tissue pieces  : {len(segmentation.tissue_polygons)}")
print(f"Holes          : {len(segmentation.hole_polygons)}")
n_islands = sum(len(p.interiors) for p in segmentation.hole_polygons)
print(f"Islands inside holes: {n_islands}")
for i, p in enumerate(sorted(segmentation.tissue_polygons, key=lambda p: -p.area)):
    print(f"  tissue[{i}] area={p.area:,.0f} um^2")

In [ ]:
ax = plot_mosaic_segmentation(canvas, segmentation)
ax.figure.set_size_inches(10, 10)
ax.figure.tight_layout()

## Write boundary_positions*.txt / hole*.txt

Run once the plot above looks right. One tissue piece is written as
`boundary_positions.txt`, several as `boundary_positions_{b}.txt`. Re-running
overwrites them.

Then continue with `03_create_positions_from_boundaries.ipynb`.

In [ ]:
written = save_boundary_from_mosaic(segmentation, OUTPUT_DIR)
print(f"Wrote {len(written)} file(s) to {OUTPUT_DIR}:")
for f in written:
    print(f"  {f}")